# 機能テスト 3: オントロジー(`ontology-build` / `ontology-consume`)

## サマリー

- オントロジーを**作る主体は、`ontology-build` の Python コードと、そこから呼ばれる Azure OpenAI のチャット モデル**です。Foundry IQ は、元のパッセージを読み出すだけです。このノートブックは、利用者が公式配布元から取得した `foundry-iq-skills` の `ontology-build` と `ontology-consume` の**機能をテストします**。ランタイムは同梱せず、自動ダウンロードも行いません。
- テストは 2 部構成です。
  - **パート A(オフライン)**: Azure もモデルも呼びません。Turtle 形式の OWL ファイルを取り込み、衛生処理(hygiene)、レビュー、承認、JSON-LD への書き出し、プロファイル検証、版の差分を確かめます。
  - **パート B(ライブ)**: Topic Graph の出力フォルダーを入力に、Azure OpenAI のチャット モデルがオントロジーを**提案**します。見積もり、承認、実行、検証までを確かめます。モデルを呼ぶため、少額の費用が発生します(見積もりは `plan` の結果に表示され、上限を `MAX_SPEND_USD` で決めます)。
- 入力のテキスト(近藤寺と花ケ咲神社の記事)は、パート B では、機能テスト 2 が作る Topic Graph の出力と、その元の Foundry IQ インデックスを通じて使われます。パート A は、同じ 2 記事の内容に合わせた小さな Turtle をノートブック内で用意します。
- `Run All` で最後まで実行すると、各テストが PASS / FAIL で記録され、最後に一覧が出ます。パート A の結果は、第 8 章の直後にも出ます。パート B が不要なら、そこで止めます。

> 出典: [Ontology build(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/ontology-build.md) / [Connect, plan, propose](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md) / [Review, approve, export](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md) / [Check and estimate](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-consume/references/check-and-estimate.md)

## テストの流れ

```mermaid
flowchart LR
    subgraph A["パート A: オフライン"]
        T["Turtle の OWL"] --> I["import<br/>衛生処理"] --> R["review"] --> P["approve"] --> E["export<br/>JSON-LD"] --> C["consume check<br/>show / diff"]
    end
    subgraph B["パート B: ライブ"]
        O["Topic Graph の<br/>出力フォルダー"] --> CN["connect"] --> PL["plan<br/>見積もり"] --> PR["propose<br/>モデル呼び出し"] --> RV["review / approve<br/>export / check"]
    end
```

## テスト一覧

| ID | 対象 | 合格条件 |
| --- | --- | --- |
| T0 | ランタイム | `setup ontology` が `ready`。CLI のバージョンが表示される |
| A1 | `import`(正常な Turtle) | 6 クラス、5 関係、1 属性が取り込まれ、削除された項目がない |
| A2 | `import`(不正な Turtle) | 宙に浮いた関係が衛生処理で削除され、`subClassOf` は無視される |
| A3 | `review` | 承認カードが出て、未承認である |
| A4 | `approve` | 誤った schema ID は `SCHEMA_ID_MISMATCH` で拒否され、正しい ID で承認できる |
| A5 | `export` | JSON-LD が書かれ、SHA-256 が一致し、同じ出力先は上書きされない |
| A6 | `consume check`(正常) | 違反が 0 件 |
| A7 | `consume check`(不正) | 3 通りの改変が、それぞれ想定した規則で拒否される |
| A8 | `consume show` / `current` | クラスと関係の一覧が一致し、現在の承認済みの SHA-256 が一致する |
| A9 | `consume diff` | 版 1 と版 2 の差分が、追加したクラスと関係だけ |
| A10 | データ型の正規化 | `xsd:integer` が `Double` に正規化される(0.1.0 での観測) |
| B1 | `doctor` / `connect` | 失敗するチェックがなく、Topic Graph の出力とソース インデックスが検証される |
| B2 | `inspect` / `plan` | トピックが見つかり、見積もりが上限以下 |
| B3 | `propose` | 新しい版ができ、費用が承認した予算以内 |
| B4 | 提案の中身 | クラスと関係があり、すべての型に根拠(evidence)が付く |
| B5 | `review` → `approve` → `export` → `check` | 提案した版が、検証を通る JSON-LD になる |

## 前提

| 項目 | 内容 |
| --- | --- |
| ランタイム | [README の手動取得・配置手順](./README.md) に従って公式配布元から取得します。既定の配置先はノートブックと同じ階層の `foundry-iq-skills` フォルダーです。別の場所なら環境変数 `FOUNDRY_IQ_PLUGIN_ROOT` で指定します |
| Python | 3.12 から 3.14 のいずれか。`setup` は、グローバルな環境を変えず、プラグイン専用のキャッシュに入れます |
| パート B | Azure CLI でサインイン済みであること。Azure OpenAI のチャット デプロイ。ソース インデックスの読み取り権限(Search Index Data Reader)。`.env` の `AZURE_OPENAI_ENDPOINT` と `AZURE_OPENAI_DEPLOYMENT` |
| パート B の入力 | 機能テスト 2 の出力(`.functional-test/topicgraph-out`)。なければ、`.foundry-iq` 配下の過去の出力を探します |

## 1. セットアップ

### 1.1 パッケージのインストール

ノートブックの側で使うのは、表示とグラフ描画のライブラリだけです。`ontology-build` 自体は、プラグインが専用の環境に入れます(次の 1.3)。

In [ ]:
%pip install -q python-dotenv pandas networkx matplotlib azure-identity

### 1.2 設定

作業フォルダー `.functional-test/ontology` を使います。**承認済みオントロジーの保存先は、環境変数 `ONTOLOGY_HOME` でこの作業フォルダーの中に分けます**。既定の保存先(`%LOCALAPPDATA%\ontology`)を、テストで汚さないためです。

`check()` は、テストの結果を `results` に記録します。通常の検証では失敗しても処理を続け、最後にまとめて表示します。ただし、B3 の提案に失敗した場合は、その応答を表示してセルを停止します。

In [ ]:
import base64
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)
matplotlib.rcParams["font.sans-serif"] = ["Yu Gothic", "Meiryo", "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "Hiragino Sans", "DejaVu Sans"]

PLUGIN_ROOT = Path(os.environ.get("FOUNDRY_IQ_PLUGIN_ROOT", "foundry-iq-skills")).expanduser().resolve()
BOOTSTRAP = PLUGIN_ROOT / "graph" / "runtime" / "bootstrap.py"
if not BOOTSTRAP.is_file():
    raise FileNotFoundError(f"Foundry IQ ランタイムが未配置です: {BOOTSTRAP}。README の手動取得・配置手順に従い、foundry-iq-skills をノートブックと同じ階層に置くか、FOUNDRY_IQ_PLUGIN_ROOT に配置先を指定してください。自動ダウンロードは行いません。")
WORK = Path(".functional-test/ontology").resolve()
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ONTOLOGY_HOME"] = str(WORK / "home")

MAX_SPEND_USD = 1.0    # パート B の提案に使う、費用の上限(ドル)
RUN_CLEANUP = False    # True にすると、最後に作業フォルダーを削除します

results = []


def check(case, ok, detail=""):
    results.append({"テスト": case, "結果": "PASS" if ok else "FAIL", "詳細": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


def cli(tool, *args, state=None):
    cmd = [sys.executable, "-I", str(BOOTSTRAP), "run", "ontology", tool, *map(str, args), "--json"]
    if state is not None:
        cmd += ["--state-dir", str(state)]
    proc = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    try:
        data = json.loads(proc.stdout)
    except ValueError:
        data = {"raw": proc.stdout, "stderr": proc.stderr[-500:]}
    return proc.returncode, data


pd.DataFrame({"値": [PLUGIN_ROOT, BOOTSTRAP.is_file(), WORK, os.environ["ONTOLOGY_HOME"]]},
             index=["プラグイン", "bootstrap.py がある", "作業フォルダー", "ONTOLOGY_HOME"])

### 1.3 T0: ランタイムの準備

`bootstrap.py setup ontology` は、固定された wheel とロック ファイルのハッシュを検証して、プラグイン専用の環境(既定は `%LOCALAPPDATA%\foundry-iq-skills\runtimes`)に入れます。Azure もモデルも呼びません。初回は pip で依存関係を取得するため、時間がかかります。`python -I` は、環境変数 `PYTHON*` や、スクリプトの場所を `sys.path` に加える動作を無効にする分離モードです。

> 出典: [Install and checks(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/graph-guide/references/install-and-checks.md) / [Python コマンドライン: -I](https://docs.python.org/3/using/cmdline.html#cmdoption-I)

In [ ]:
proc = subprocess.run([sys.executable, "-I", str(BOOTSTRAP), "setup", "ontology"], capture_output=True, text=True, encoding="utf-8", errors="replace")
setup = json.loads(proc.stdout.strip().splitlines()[-1])
check("T0-1 ontology ランタイムの準備", proc.returncode == 0 and setup.get("status") == "ready", setup.get("python"))

version = subprocess.run([sys.executable, "-I", str(BOOTSTRAP), "run", "ontology", "ontology-build", "--version"],
                         capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
check("T0-2 ontology-build のバージョンが表示される", version.startswith("ontology-build "), version)

## パート A: オフラインのテスト

## 2. 入力の用意(Turtle の OWL)

OWL 2 は、クラス、プロパティ、個体、データ値を持つオントロジー言語で、RDF 文書として交換されます。Turtle は、その RDF を人が書きやすくした形式です。`ontology-build import` は、`.owl` `.rdf` `.xml` `.ttl` `.nt` `.json` を読めます(`rdflib` などで解析します)。

2 つの入力を用意します。

| ファイル | 内容 | 狙い |
| --- | --- | --- |
| `hanapedia.ttl` | 6 クラス、5 つのオブジェクト プロパティ、1 つのデータ型プロパティ | 正常に取り込めること |
| `bad.ttl` | `subClassOf` と、存在しないクラスを指す `range` を含む | 衛生処理(hygiene)で、プロファイルに合わない項目が除かれること |

> 出典: [OWL 2 Web Ontology Language Document Overview(W3C)](https://www.w3.org/TR/owl2-overview/) / [Review, approve, export(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md)

In [ ]:
FIXTURES = WORK / "fixtures"
FIXTURES.mkdir(parents=True, exist_ok=True)

PREFIXES = """@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix : <https://example.com/hanapedia#> .

<https://example.com/hanapedia> a owl:Ontology ;
    rdfs:label "Hanapedia" .
"""

GOOD_TTL = PREFIXES + """
:ReligiousSite a owl:Class ; rdfs:label "ReligiousSite" ; rdfs:comment "神社や寺院などの宗教施設。例: 花ケ咲神社、近藤寺" .
:Deity a owl:Class ; rdfs:label "Deity" ; rdfs:comment "祭神や本尊として祀られる神仏。例: 桜花女神" .
:Person a owl:Class ; rdfs:label "Person" ; rdfs:comment "人物。例: 近藤智海、織田信長" .
:Facility a owl:Class ; rdfs:label "Facility" ; rdfs:comment "境内の建造物や庭園。例: 本殿、金堂" .
:Place a owl:Class ; rdfs:label "Place" ; rdfs:comment "地名や行政区画。例: 京都市東山区" .
:Event a owl:Class ; rdfs:label "Event" ; rdfs:comment "祭事や歴史上の出来事。例: 桜花祭、応仁の乱" .

:locatedIn a owl:ObjectProperty ; rdfs:label "locatedIn" ; rdfs:domain :ReligiousSite ; rdfs:range :Place ; rdfs:comment "宗教施設が所在する地域" .
:enshrines a owl:ObjectProperty ; rdfs:label "enshrines" ; rdfs:domain :ReligiousSite ; rdfs:range :Deity ; rdfs:comment "宗教施設が祀る祭神や本尊" .
:foundedBy a owl:ObjectProperty ; rdfs:label "foundedBy" ; rdfs:domain :ReligiousSite ; rdfs:range :Person ; rdfs:comment "宗教施設を創建した人物" .
:hasFacility a owl:ObjectProperty ; rdfs:label "hasFacility" ; rdfs:domain :ReligiousSite ; rdfs:range :Facility ; rdfs:comment "境内にある建造物や庭園" .
:survivedEvent a owl:ObjectProperty ; rdfs:label "survivedEvent" ; rdfs:domain :ReligiousSite ; rdfs:range :Event ; rdfs:comment "焼失を免れた出来事" .
:foundedYear a owl:DatatypeProperty ; rdfs:label "foundedYear" ; rdfs:domain :ReligiousSite ; rdfs:range xsd:integer ; rdfs:comment "創建された西暦年" .
"""

BAD_TTL = """@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix : <https://example.com/bad#> .

<https://example.com/bad> a owl:Ontology .

:Site a owl:Class ; rdfs:comment "施設" .
:Temple a owl:Class ; rdfs:subClassOf :Site ; rdfs:comment "寺" .
:Place a owl:Class ; rdfs:comment "場所" .
:locatedIn a owl:ObjectProperty ; rdfs:domain :Site ; rdfs:range :Missing ; rdfs:comment "所在" .
"""

V2_TTL = GOOD_TTL + """
:Artifact a owl:Class ; rdfs:label "Artifact" ; rdfs:comment "寺社が所蔵する文化財" .
:hasArtifact a owl:ObjectProperty ; rdfs:label "hasArtifact" ; rdfs:domain :ReligiousSite ; rdfs:range :Artifact ; rdfs:comment "所蔵する文化財" .
"""

for name, text in {"hanapedia.ttl": GOOD_TTL, "bad.ttl": BAD_TTL, "hanapedia-v2.ttl": V2_TTL}.items():
    (FIXTURES / name).write_text(text, encoding="utf-8")
pd.DataFrame([{"ファイル": p.name, "バイト数": p.stat().st_size} for p in sorted(FIXTURES.glob("*.ttl"))])

## 3. A1 / A2: `import` と衛生処理

`import` は、ファイルを読み、プロファイルに合わない項目を**除く、または名前を直す**衛生処理(hygiene)をかけて、新しい版(`version`)として保存します。結果の `hygiene.dropped` が、除かれた項目の数です。`status` が `ok` でも、中身が減っている場合があるため、件数を確かめます。

状態フォルダー(`--state-dir`)は、版、承認、書き出しの記録を持つフォルダーです。テストごとに作り直します。

In [ ]:
STATE_A = WORK / "state-a"
STATE_BAD = WORK / "state-bad"
for folder in (STATE_A, STATE_BAD):
    shutil.rmtree(folder, ignore_errors=True)

rc, good = cli("ontology-build", "import", FIXTURES / "hanapedia.ttl", state=STATE_A)
check("A1 正常な Turtle を取り込める", rc == 0 and (good["classes"], good["relationships"], good["attributes"]) == (6, 5, 1) and good["hygiene"]["dropped"] == 0,
      f"クラス {good.get('classes')} / 関係 {good.get('relationships')} / 属性 {good.get('attributes')} / 削除 {good.get('hygiene', {}).get('dropped')}")

rc, bad = cli("ontology-build", "import", FIXTURES / "bad.ttl", state=STATE_BAD)
check("A2 不正な項目が衛生処理で除かれる", rc == 0 and bad["relationships"] == 0 and bad["hygiene"]["dropped"] == 1 and bad["classes"] == 3,
      f"クラス {bad.get('classes')} / 関係 {bad.get('relationships')} / 削除 {bad.get('hygiene', {}).get('dropped')}")

pd.DataFrame({"正常な入力": {k: good.get(k) for k in ("version", "classes", "relationships", "attributes", "schema_id")},
              "不正な入力": {k: bad.get(k) for k in ("version", "classes", "relationships", "attributes", "schema_id")}})

## 4. A3 / A4: レビューと承認

`review` は、版の中身を平易な件数(型、関係、属性)で示し、**承認カード**を返します。承認は Azure への書き込みではなく、「この版を以降の手順が使う、現在のオントロジーにする」というレビューです。

`approve` には、`review` が示した **schema ID**(内容のハッシュ)を渡します。ID が違えば `SCHEMA_ID_MISMATCH` で拒否されます。レビューした内容と、承認する内容が、すり替わらないための仕組みです。

In [ ]:
rc, review = cli("ontology-build", "review", "--version", 1, state=STATE_A)
check("A3 レビューで承認カードが出て、未承認である", rc == 0 and review["approved"] is False and review["approval"][0].startswith("APPROVAL"),
      review["approval"][1] if rc == 0 else review)
print("\n".join(review["approval"]))

SCHEMA_ID = review["schema_id"]
rc, wrong = cli("ontology-build", "approve", "--version", 1, "--schema-id", "sha256:deadbeef", state=STATE_A)
check("A4-1 誤った schema ID は拒否される", rc != 0 and wrong.get("error_code") == "SCHEMA_ID_MISMATCH", wrong.get("error_code"))

rc, approved = cli("ontology-build", "approve", "--version", 1, "--schema-id", SCHEMA_ID, state=STATE_A)
check("A4-2 正しい schema ID で承認できる", rc == 0 and approved["status"] == "ok" and approved["already_approved"] is False, approved.get("approved_at"))

## 5. A5: JSON-LD への書き出し

`export` は、**承認済みの版だけ**を、OWL の JSON-LD(`.jsonld`)に書き出します。これが Entity Graph の `--ontology inline_jsonld --ontology-path` に渡す形式です。次を確かめます。

- 出力は、正規化された(canonical)JSON です。同じ内容なら、同じバイト列になります。
- 結果に記録された SHA-256 が、ファイルから計算した値と一致します。
- 既存のファイルは、`--force` を付けない限り上書きしません(`OUTPUT_EXISTS`)。

> 出典: [JSON-LD 1.1(W3C)](https://www.w3.org/TR/json-ld11/) / [Review, approve, export(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/review-approve-export.md)

In [ ]:
OUT = WORK / "out"
shutil.rmtree(OUT, ignore_errors=True)
JSONLD = OUT / "hanapedia.jsonld"

rc, exported = cli("ontology-build", "export", "--version", 1, "--output", JSONLD, state=STATE_A)
actual_sha = hashlib.sha256(JSONLD.read_bytes()).hexdigest() if JSONLD.is_file() else None
check("A5-1 JSON-LD が書かれ、SHA-256 が一致する", rc == 0 and exported["profile"] == "inline-jsonld-v1" and actual_sha == exported["jsonld_sha256"], exported.get("profile"))

rc, again = cli("ontology-build", "export", "--version", 1, "--output", JSONLD, state=STATE_A)
check("A5-2 同じ出力先は上書きされない", rc != 0 and again.get("error_code") == "OUTPUT_EXISTS", again.get("error_code"))

print(JSONLD.read_text(encoding="utf-8")[:600], "...")

## 6. A6 / A7: プロファイル検証(`ontology-consume check`)

`check` は、JSON-LD(または `.owl` `.rdf` `.xml`)が**オントロジーのプロファイル**に合っているかを調べ、違反を規則名つきで返します。Entity Graph のナレッジ ソースへ接続する前の、事前検査に使えます。主な規則は、ファイルが正規化されていること、宣言が `owl:Class` `owl:ObjectProperty` `owl:DatatypeProperty` のいずれかであること、プロパティの `domain` と `range` が宣言済みのクラスを指すこと、です。Turtle は `check` の対象外で、`import` してから `export` します。

正常なファイルに、次の 3 通りの改変を加えて、それぞれ拒否されることを確かめます。

| 改変 | 期待する規則 |
| --- | --- |
| プロパティの `range` を、存在しないクラスにする | `property_range` |
| プロパティの型を `owl:Restriction` にする | `declaration_type` |
| 整形して、バイト列を正規形から外す | `not_canonical` |

> 出典: [Check and estimate(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-consume/references/check-and-estimate.md)

In [ ]:
rc, ok = cli("ontology-consume", "check", JSONLD)
check("A6 正常な JSON-LD は違反が 0 件", rc == 0 and ok["status"] == "ok" and ok["violations"] == [], ok.get("violations"))

original = json.loads(JSONLD.read_text(encoding="utf-8"))


def dump(obj, pretty=False):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, **({"indent": 2} if pretty else {"separators": (",", ":")}))


dangling = json.loads(json.dumps(original))
next(n for n in dangling["@graph"] if n["@id"].endswith("#enshrines"))["rdfs:range"] = {"@id": "urn:ontology:ontology#Ghost"}

wrong_type = json.loads(json.dumps(original))
next(n for n in wrong_type["@graph"] if n["@id"].endswith("#enshrines"))["@type"] = "owl:Restriction"

NEGATIVES = {"property_range": dump(dangling), "declaration_type": dump(wrong_type), "not_canonical": dump(original, pretty=True)}
rows = []
for rule, text in NEGATIVES.items():
    path = OUT / f"negative-{rule}.jsonld"
    path.write_text(text, encoding="utf-8")
    rc, data = cli("ontology-consume", "check", path)
    violations = data.get("details", {}).get("violations", [])
    hit = any(v.startswith(rule) for v in violations)
    check(f"A7 {rule} の違反を検出する", rc != 0 and data.get("error_code") == "PROFILE_VIOLATION" and hit, violations[:1])
    rows.append({"改変": rule, "error_code": data.get("error_code"), "違反": " / ".join(violations)})
pd.DataFrame(rows)

## 7. A8: 内容の閲覧と「現在の承認済み」

`ontology-consume show` は、クラスとプロパティの一覧を返し、`current` は、**現在承認されているオントロジー**のパス、SHA-256、件数を返します。`current` は呼ぶたびにファイルのハッシュを計算し直すため、ファイルが書き換えられていれば `CURRENT_INVALID` で止まります。

取り込み時の宣言を、クラスを丸、オブジェクト プロパティを矢印にして図にします。

In [ ]:
rc, shown = cli("ontology-consume", "show", "--version", 1, state=STATE_A)
class_names = {c["name"] for c in shown["classes"]}
relations = {(p["domain"], p["name"], p["range"]) for p in shown["properties"] if p["kind"] == "object"}
EXPECTED_CLASSES = {"ReligiousSite", "Deity", "Person", "Facility", "Place", "Event"}
EXPECTED_RELATIONS = {("ReligiousSite", "locatedIn", "Place"), ("ReligiousSite", "enshrines", "Deity"), ("ReligiousSite", "foundedBy", "Person"),
                      ("ReligiousSite", "hasFacility", "Facility"), ("ReligiousSite", "survivedEvent", "Event")}
check("A8-1 クラスと関係が入力と一致する", rc == 0 and class_names == EXPECTED_CLASSES and relations == EXPECTED_RELATIONS, f"{len(class_names)} クラス / {len(relations)} 関係")

rc, current = cli("ontology-consume", "current", state=STATE_A)
current_sha = hashlib.sha256(Path(current["path"]).read_bytes()).hexdigest()
check("A8-2 現在の承認済みの SHA-256 が一致する", rc == 0 and current["sha256"] == current_sha == actual_sha, current["sha256"][:16])


def draw_ontology(show, title):
    g = nx.DiGraph()
    g.add_nodes_from(c["name"] for c in show["classes"])
    g.add_edges_from((p["domain"], p["range"], {"predicate": p["name"]}) for p in show["properties"] if p["kind"] == "object")
    pos = nx.spring_layout(g, seed=7, k=1.4)
    fig, ax = plt.subplots(figsize=(9, 6))
    nx.draw_networkx_nodes(g, pos, ax=ax, node_size=1800, node_color="#9ecae1")
    nx.draw_networkx_edges(g, pos, ax=ax, arrows=True, arrowsize=14, edge_color="#888888", node_size=1800)
    nx.draw_networkx_labels(g, pos, ax=ax, font_size=8)
    nx.draw_networkx_edge_labels(g, pos, ax=ax, edge_labels=nx.get_edge_attributes(g, "predicate"), font_size=7)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


draw_ontology(shown, "取り込んだオントロジー(版 1)")

## 8. A9 / A10: 版の差分とデータ型の正規化

同じ状態フォルダーに、クラス `Artifact` と関係 `hasArtifact` を足した Turtle を**版 2**として取り込み、`diff` が差分だけを返すことを確かめます。版は上書きされず、追記されます。

あわせて、Turtle で `xsd:integer` と書いた `foundedYear` が、プロファイルの上では `Double` として扱われることを確かめます。プロファイルは、データ型の範囲を `String` `Double` `Boolean` `DateTime` に揃えているためと考えられます(`ontology-build` の提案用プロンプトにも、同じ 4 つが書かれています)。これは 0.1.0 での観測です。

> 出典: [prompts.py(ontology_skills ホイール)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
rc, v2 = cli("ontology-build", "import", FIXTURES / "hanapedia-v2.ttl", state=STATE_A)
rc, diff = cli("ontology-consume", "diff", 1, 2, state=STATE_A)
added = diff["diff"]["added"]
check("A9 版の差分が追加分だけ", rc == 0 and v2["version"] == 2 and added == {"classes": ["Artifact"], "properties": ["hasArtifact"]}
      and diff["diff"]["removed"] == {"classes": [], "properties": []}, added)

rc, shown2 = cli("ontology-consume", "show", "--version", 2, state=STATE_A)
founded_year = next(p for p in shown2["properties"] if p["name"] == "foundedYear")
check("A10 xsd:integer が Double に正規化される", founded_year["range"] == "Double", f"kind={founded_year['kind']} range={founded_year['range']}")
draw_ontology(shown2, "版 2(Artifact を追加)")

### パート A の結果

ここまでのテストの結果です。パート B(Azure とモデルを使う)は、次の章から始まります。

In [ ]:
part_a = pd.DataFrame(results)
print(part_a["結果"].value_counts().to_dict())
part_a

## パート B: ライブのテスト(Azure とモデルを使う)

## 9. 入力と接続先を決める

パート B は、`ontology-build` が**実際にオントロジーを提案する**ところを確かめます。流れは、次の 5 つの動詞です。

| 動詞 | Azure | モデル | 内容 |
| --- | --- | --- | --- |
| `connect` | 読み取りのみ | 呼ばない | 状態フォルダーを、テナント、Topic Graph の出力フォルダー、モデルのデプロイに結ぶ。ソース インデックスのスキーマを読み、固定する |
| `inspect` | 呼ばない | 呼ばない | トピックのレベルと、既定のサンプリングを一覧する |
| `plan` | 呼ばない | 呼ばない | 呼び出し回数と費用を**見積もり**、計画を保存する。承認カードを返す |
| `propose` | 読み取りのみ | **呼ぶ** | 承認された計画を実行する。選んだパッセージを読み、チャット モデルに提案させる |
| `review` / `approve` / `export` | 呼ばない | 呼ばない | 提案を、パート A と同じ手順で確定する |

入力の `--topic-graph-output` は、Topic Graph の実行ファイルが `outputDirectory` に書いたフォルダーです。**元のソース インデックスの場所は、このフォルダーの記録(`context.json`)から読みます**。パッセージの本文は、ビルド時ではなく、**現在の**インデックスから読みます(`--accept-current-source` はその承諾です)。

> 出典: [Connect, plan, propose(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md)

In [ ]:
from azure.identity import DefaultAzureCredential

AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

token = DefaultAzureCredential().get_token("https://management.azure.com/.default").token
payload = token.split(".")[1]
TENANT_ID = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))["tid"]

# モデルの単価(100 万トークンあたりのドル)。費用の上限を守るために必要です。環境変数、なければ過去のビルド記録から読みます
PRICE_IN = os.environ.get("MODEL_INPUT_PRICE_PER_1M")
PRICE_OUT = os.environ.get("MODEL_OUTPUT_PRICE_PER_1M")
recorded = next(iter(sorted(Path(".foundry-iq").glob("*/entity/build-args.json"))), None)
if (PRICE_IN is None or PRICE_OUT is None) and recorded:
    PRICE_IN, PRICE_OUT = json.loads(recorded.read_text(encoding="utf-8"))["chat_model_usd_per_1m"]

candidates = [Path(".functional-test/topicgraph-out"), *sorted(Path(".foundry-iq").glob("*/topics/topicgraph-output/*"))]
TOPICGRAPH_OUTPUT = next((p.resolve() for p in candidates if (p / "topicGraph" / "run-manifest.json").is_file()), None)

assert TOPICGRAPH_OUTPUT, "Topic Graph の出力が見つかりません。機能テスト 2 を先に実行するか、TOPICGRAPH_OUTPUT にパスを指定してください"
assert PRICE_IN is not None and PRICE_OUT is not None, "モデルの単価が不明です。環境変数 MODEL_INPUT_PRICE_PER_1M と MODEL_OUTPUT_PRICE_PER_1M を設定してください"

pd.DataFrame({"値": [AOAI_ENDPOINT, CHAT_DEPLOYMENT, TOPICGRAPH_OUTPUT, PRICE_IN, PRICE_OUT]},
             index=["Azure OpenAI", "チャット デプロイ", "Topic Graph の出力", "入力単価(ドル / 100 万トークン)", "出力単価(ドル / 100 万トークン)"])

## 10. B1: `doctor` と `connect`

`doctor` は読み取り専用で、インストール、サインイン、Topic Graph の出力、ソース インデックス、モデルのアクセスを順に調べます。**モデルは呼びません**。「モデルのチャット アクセスは、最初の承認された呼び出しまで未確認」と表示されるのは、正常です。

`connect` が通ると、状態フォルダーに接続が保存されます。以降の動詞は、`--state-dir` だけで同じ接続を使います。

In [ ]:
STATE_B = WORK / "state-live"
shutil.rmtree(STATE_B, ignore_errors=True)

MODEL_FLAGS = ["--model-endpoint", AOAI_ENDPOINT, "--model-deployment", CHAT_DEPLOYMENT]
rc, doctor = cli("ontology-build", "doctor", "--tenant-id", TENANT_ID, *MODEL_FLAGS, state=STATE_B)
check("B1-1 doctor に失敗するチェックがない", rc == 0 and all(c["status"] != "fail" for c in doctor["checks"]), [c["name"] for c in doctor["checks"] if c["status"] == "fail"])
display(pd.DataFrame([{"チェック": c["name"], "状態": c["status"], "詳細": c["detail"][:110]} for c in doctor["checks"]]))

In [ ]:
rc, connected = cli("ontology-build", "connect", "--tenant-id", TENANT_ID, "--topic-graph-output", TOPICGRAPH_OUTPUT, "--accept-current-source",
                    *MODEL_FLAGS, "--model-input-price-per-1m", PRICE_IN, "--model-output-price-per-1m", PRICE_OUT, state=STATE_B)
by_name = {c["name"]: c["status"] for c in connected["checks"]}
check("B1-2 connect が成功し、ソース インデックスとトピックが検証される",
      rc == 0 and connected.get("status") == "ok" and by_name.get("source_index") == "ok" and by_name.get("topicgraph_topics") == "ok",
      connected.get("source_index"))
pd.DataFrame([{"チェック": c["name"], "状態": c["status"], "詳細": c["detail"][:110]} for c in connected["checks"]])

## 11. B2: `inspect` と `plan`(見積もり)

`inspect` は、トピックのレベルごとの数を返します。**レベル 0 が最も広く**、Topic Graph と同じ番号です。`plan` の既定のレベルは、トピックが 20 以上ある最も広いレベルです。トピックが 1 つしかない小さな出力では、レベル 0 が選ばれました(過去の出力での観測)。

`plan` は、モデルを呼ばずに、呼び出し回数と費用を見積もり、計画 ID を保存します。見積もりの `usd` は、**すべてのトークンを入出力の高いほうの単価で見積もった最悪値**で、`expected_usd` が入出力を別々の単価で見積もった期待値です。最悪値が上限を超えると、`BUDGET_CAP_BELOW_ESTIMATE` で止まります。

In [ ]:
rc, inspected = cli("ontology-build", "inspect", state=STATE_B)
check("B2-1 トピックのレベルが見つかる", rc == 0 and len(inspected["levels"]) >= 1 and inspected["levels"][0].get("topics", 0) >= 1,
      f"レベル数 {len(inspected['levels'])} / 既定のレベル {inspected.get('default_level')} / 既定のパッセージ数 {inspected.get('default_passages')}")
display(pd.DataFrame(inspected["levels"]))

rc, plan = cli("ontology-build", "plan", "--model-max-spend-usd", MAX_SPEND_USD, state=STATE_B)
check("B2-2 見積もりが上限以下", rc == 0 and plan["estimate"]["usd"] <= MAX_SPEND_USD,
      f"最悪 ${plan['estimate']['usd']:.4f} / 期待 ${plan['estimate'].get('expected_usd', 0):.4f} / 呼び出し {plan.get('calls')} 回")
print("\n".join(plan["approval"]))

## 12. B3: `propose`(モデルを呼ぶ)

**このセルは、Azure OpenAI のチャット モデルを呼びます。** `--approved-budget-usd` に、見積もりの最悪値を切り上げた額を渡し、`--approve-model-calls` で、モデル呼び出しを承認します。この 2 つのフラグが、承認カードへの Approve に当たります。上限を超える呼び出しは、実行されません。

内部では、2 種類のシステム プロンプトが使われます。

1. **提案(PROPOSAL)**: トピックごとにまとめたパッセージを見て、クラスと関係を、根拠の ID つきで提案させます。
2. **監査(AUDIT)**: 分割して得た候補を、根拠と突き合わせて 1 つに統合させます(同義語の統合、広すぎる型の除去など)。

`RUN_INCOMPLETE` とモデル接続の `SSLError` が返った場合だけ、2 秒、4 秒待って最大 2 回、同じ計画 ID・状態ディレクトリ・予算で再開します。証明書検証は無効にしません。他のエラーや再試行上限では、終了コードと元の CLI 応答を含む `RuntimeError` でセルを停止します。成功時にだけ返る `spent_usd` などは、失敗時には参照しません。

原因を解消した後、状態ディレクトリと計画 ID を変えずに、この B3 セルを再実行します。完了した呼び出しは、再送も再課金もされません。B1 からやり直すと状態ディレクトリを削除するため、再開時には実行しないでください。

> 出典: [Connect, plan, propose(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/ontology-build/references/connect-plan-propose.md) / [prompts.py(ontology_skills ホイール)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels) / [Python の組み込み例外: RuntimeError](https://docs.python.org/3/library/exceptions.html#RuntimeError)

In [ ]:
import time

BUDGET = math.ceil(plan["estimate"]["usd"] * 100) / 100
for attempt in range(3):
    rc, proposed = cli("ontology-build", "propose", "--plan-id", plan["plan_id"], "--approved-budget-usd", BUDGET, "--approve-model-calls", state=STATE_B)
    tls_failed = proposed.get("error_code") == "RUN_INCOMPLETE" and proposed.get("error", "").endswith(": SSLError)")
    if rc == 0 or not tls_failed or attempt == 2:
        break
    delay = 2 ** (attempt + 1)
    print(f"モデル接続の TLS エラーのため、{delay} 秒後に同じ計画で再開します ({attempt + 1}/2)。")
    time.sleep(delay)

if not check("B3-1 新しい版ができる", rc == 0 and proposed.get("status") == "ok" and proposed.get("version", 0) >= 1, f"版 {proposed.get('version')}"):
    raise RuntimeError(f"ontology-build propose が失敗しました (終了コード {rc}):\n{json.dumps(proposed, ensure_ascii=False, indent=2)}")
check("B3-2 費用が承認した予算以内", proposed["spent_usd"] <= proposed["approved_budget_usd"], f"${proposed['spent_usd']:.4f} / ${proposed['approved_budget_usd']}")
pd.DataFrame([{k: proposed.get(k) for k in ("version", "classes", "properties", "passages", "requested_passages", "spent_usd", "dropped", "warnings")}])

In [ ]:
rc, proposed = cli("ontology-build", "review", "--version", 1, state=STATE_B)

## 13. B4: 提案の中身を確かめる

提案された型は、すべて**根拠(evidence)の ID を 1 〜 5 個**持つ決まりです。根拠が 1 つもない型は、衛生処理で除かれます。`show` の `evidence` は、各型の根拠の件数です。クラスとプロパティを一覧し、図にします。

In [ ]:
rc, live_shown = cli("ontology-consume", "show", "--version", proposed["version"], "--limit", 200, state=STATE_B)
classes_df = pd.DataFrame(live_shown["classes"], columns=["name", "evidence", "description"])
props_df = pd.DataFrame(live_shown["properties"], columns=["domain", "name", "range", "kind", "evidence"])

check("B4-1 クラスが 3 つ以上ある", len(classes_df) >= 3, f"{len(classes_df)} クラス")
check("B4-2 関係(プロパティ)が 2 つ以上ある", len(props_df) >= 2, f"{len(props_df)} プロパティ")
check("B4-3 すべての型に根拠が付く", bool((classes_df["evidence"] >= 1).all() and (props_df["evidence"] >= 1).all()),
      f"根拠なし: クラス {int((classes_df['evidence'] < 1).sum())} / プロパティ {int((props_df['evidence'] < 1).sum())}")
display(classes_df)
display(props_df)
draw_ontology(live_shown, "提案されたオントロジー")

### 根拠のパッセージを見る

`ontology-consume evidence <型の名前>` は、その型の根拠になったパッセージを返します。提案が、原文のどこから出てきたかを、人が確かめるための動詞です。

In [ ]:
rc, evidence = cli("ontology-consume", "evidence", classes_df["name"].iloc[0], "--limit", 3, state=STATE_B)
print(json.dumps(evidence, ensure_ascii=False, indent=2)[:2000])

## 14. B5: 確定して、プロファイル検証を通す

パート A と同じ手順で、`review`、`approve`、`export` を行い、書き出した JSON-LD を `check` に通します。**生成された版が、そのまま Entity Graph に渡せる形式であること**を確かめるテストです。

In [ ]:
VERSION = proposed["version"]
LIVE_JSONLD = OUT / "proposed.jsonld"
rc, live_review = cli("ontology-build", "review", "--version", VERSION, state=STATE_B)
rc, live_approved = cli("ontology-build", "approve", "--version", VERSION, "--schema-id", live_review["schema_id"], state=STATE_B)
rc, live_export = cli("ontology-build", "export", "--version", VERSION, "--output", LIVE_JSONLD, state=STATE_B)
rc, live_check = cli("ontology-consume", "check", LIVE_JSONLD)

check("B5-1 提案した版を承認して書き出せる", live_approved["status"] == "ok" and live_export["profile"] == "inline-jsonld-v1", live_export.get("output"))
check("B5-2 書き出した JSON-LD がプロファイル検証を通る", rc == 0 and live_check["status"] == "ok" and live_check["violations"] == [], live_check.get("violations"))
print("次の手順: entity-graph-build の --ontology inline_jsonld --ontology-path に", LIVE_JSONLD, "を渡します")

## 15. 結果の一覧

PASS と FAIL の件数と、FAIL の詳細を表示します。パート B を実行しなかった場合は、パート B の行は記録されません。

In [ ]:
summary = pd.DataFrame(results)
print(summary["結果"].value_counts().to_dict())
summary.style.apply(lambda row: ["background-color: #ffd6d6" if row["結果"] == "FAIL" else "" for _ in row], axis=1)

## 16. 後片付け(任意)

作業フォルダー(状態フォルダー、書き出したファイル、`ONTOLOGY_HOME`)を削除します。`ontology` ランタイム(プラグイン専用のキャッシュ)は、他の手順でも使うため、削除しません。Azure には何も作っていないので、Azure 側の後片付けはありません。

In [ ]:
if RUN_CLEANUP:
    shutil.rmtree(WORK, ignore_errors=True)
    print("削除しました:", WORK)